In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay
)
import matplotlib.pyplot as plt
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from helpers.data_loader import DataLoader

In [ ]:
train = pd.read_parquet(DataLoader.transformed('train_t03.parquet'))
val   = pd.read_parquet(DataLoader.transformed('val_t03.parquet'))
test  = pd.read_parquet(DataLoader.transformed('test_t03.parquet'))

print(f'Train: {train.shape}  Val: {val.shape}  Test: {test.shape}')

In [ ]:
TARGET = 'Results'

X_train, y_train = train.drop(columns=TARGET), train[TARGET]
X_val,   y_val   = val.drop(columns=TARGET),   val[TARGET]
X_test,  y_test  = test.drop(columns=TARGET),  test[TARGET]

for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    ratio = (y==0).sum() / (y==1).sum()
    print(f'{name}: {y.value_counts(normalize=True).mul(100).round(1).to_dict()}  ratio={ratio:.2f}:1')

In [ ]:
# Pipeline ensures scaler is fit on train only — no leakage
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)

In [ ]:
# for C in [0.01, 0.1, 1, 10, 100]:
#     p = Pipeline([
#         ('scaler', StandardScaler()),
#         ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, C=C))
#     ])
#     p.fit(X_train, y_train)
#     proba = p.predict_proba(X_val)[:, 1]
#     print(f'C={C:6}  val ROC-AUC={roc_auc_score(y_val, proba):.4f}')

In [ ]:
def evaluate(pipeline, X, y, split_name):
    y_pred       = pipeline.predict(X)
    y_pred_proba = pipeline.predict_proba(X)[:, 1]
    cm           = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f'=== {split_name} ===')
    print(classification_report(y, y_pred, target_names=['Pass', 'Fail']))
    print(f'ROC-AUC:                        {roc_auc_score(y, y_pred_proba):.4f}')
    print(f'False Negative Rate (missed fails): {fn/(fn+tp):.4f}')
    print(f'False Positive Rate (wasted inspections): {fp/(fp+tn):.4f}')
    print()
    return y_pred_proba

evaluate(pipeline, X_train, y_train, 'Train')
val_proba = evaluate(pipeline, X_val, y_val, 'Validation')

In [ ]:
# Only run this cell when you are done tuning and ready for final reporting
test_proba = evaluate(pipeline, X_test, y_test, 'Test')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

RocCurveDisplay.from_predictions(y_val, val_proba, ax=axes[0], name='Logistic Regression')
axes[0].set_title('ROC Curve — Validation')

coef_df = pd.DataFrame({
    'feature':     X_train.columns,
    'coefficient': pipeline.named_steps['model'].coef_[0]
}).sort_values('coefficient')

axes[1].barh(coef_df['feature'], coef_df['coefficient'])
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Feature Coefficients')
axes[1].set_xlabel('Coefficient value')

plt.tight_layout()

# to save the figure
# plt.savefig('logistic_regression_results.png', dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
import joblib

models_dir = Path.cwd().parent.parent / 'models'
models_dir.mkdir(exist_ok=True)

joblib.dump(pipeline, models_dir / 'logistic_regression.pkl')
print(f'Model saved to {models_dir / "logistic_regression.pkl"}')